Install required libraries

In [1]:
!pip install -U langchain==0.2.14 langchain-community langchain-core langchain-google-genai wikipedia

  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_core-1.2.17-py3-none-any.whl.metadata (4.4 kB)
  Using cached langchain_google_genai-4.2.1-py3-none-any.whl.metadata (2.7 kB)
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_community-0.4-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_community-0.3.31-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_community-0.3.30-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_community-0.3.29-py3-none-any.whl.metadata (2.9 kB)
  Using cached langchain_community-0.3.28-py3-none-any.whl.metadata (2.9 kB)
  Using cached langchain_community-0.3.27-py3-none-any.whl.metadata (2.9 kB)
  Using cached langchain_community-0.3.26-py3-none-any.whl.metadata (2.9 kB)
INFO: pip is still looking at multiple versions of langchain-community to determi

Import Required Libraries

In [2]:
import os
import wikipedia

from langchain_core.tools import Tool
from langchain.agents import AgentExecutor, create_react_agent
from langchain import hub

from langchain_google_genai import ChatGoogleGenerativeAI

Fetch API key

In [27]:
from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

Create the Calculator Tool

In [4]:
def calculator_tool(expression: str):
    try:
        result = eval(expression)
        return str(result)
    except:
        return "Error in calculation"

Create the Search Tool

In [5]:
def search_tool(query: str):
    try:
        result = wikipedia.summary(query, sentences=1)
        return result
    except:
        return "No result found"

Convert Functions into LangChain Tools

In [6]:
calculator = Tool(
    name="CalculatorTool",
    func=calculator_tool,
    description="Useful for performing mathematical calculations like add, subtract, multiply, divide."
)

search = Tool(
    name="SearchTool",
    func=search_tool,
    description="Useful for searching factual information like dates, history, or people."
)

tools = [calculator, search]

Initialize the LLM

In [40]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0
)

Create the Self-Correcting Agent

In [41]:
from langchain.agents import initialize_agent, AgentType

In [42]:
agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

Run the Assignment Prompt

In [45]:
query = "Multiply the birth year of Albert Einstein by 5."

result = agent.run(query)

print("\nFinal Answer:", result)



> Entering new AgentExecutor chain...
Action: SearchTool
Action Input: Albert Einstein birth year
Observation: Hans Albert Einstein (May 14, 1904 – July 26, 1973) was a Swiss-American engineer, the second child and first son of physicists Albert Einstein and Mileva Marić.
Thought:Action: SearchTool
Action Input: Albert Einstein's birth year
Observation: Hans Albert Einstein (May 14, 1904 – July 26, 1973) was a Swiss-American engineer, the second child and first son of physicists Albert Einstein and Mileva Marić.
Thought:Action: SearchTool
Action Input: Albert Einstein birth date
Observation: Albert Einstein (14 March 1879 – 18 April 1955) was a German-born theoretical physicist best known for developing the theory of relativity.
Thought:Action: CalculatorTool
Action Input: 1879 * 5
Observation: 9395
Thought:I now know the final answer
Final Answer: 9395

> Finished chain.

Final Answer: 9395
